# Penalty Nullified Plays
When a penalty wipes out a play, the yards and touchdowns don't count and only survive in the play description. I parsed the descriptions to recover what each player would have earned, and stored it in the ledger.

In [1]:
import os, re, numpy as np, pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 200)

CURRENT_SZN = 2026

SCORING = {
    "passing_yards": 0.04,
    "passing_tds": 4,
    "interceptions": -2,
    "rushing_yards": 0.1,
    "rushing_tds": 6,
    "receptions": 1,
    "receiving_yards": 0.1,
    "receiving_tds": 6,
    "two_pt_conversions": 2,
    "fumbles_lost": -2,
    "special_teams_tds": 6,
}

pbp = pd.concat([pd.read_parquet(f"data/pbp_{CURRENT_SZN}.parquet")], ignore_index = True)
pbp = pbp[pbp["season_type"] == "REG"].copy()

# A nullified play is "no_play" with a penalty. Timeouts and other no_plays have penalty = 0.
null = pbp[(pbp["play_type"] == "no_play") & (pbp["penalty"] == 1)].copy()
print(f"Nullified plays: {len(null):,}")

Nullified plays: 1,679


## Parsing the Description
Each description has two parts: what happened on the field, then the penalty that erased it. Only the first part matters for fantasy points, so it is split off at the word "PENALTY". From there, the play is sorted into a type (completion, run, sack, and so on), and the yards are pulled from the phrase "for N yards" or "for no gain".

nflfastR already fills in `passer_id`, `receiver_id`, and `rusher_id` on these plays from the description, so player names do not have to be matched by hand. On scrambles it fills in `passer_id` instead of `rusher_id`, which the code accounts for.

In [2]:
def on_field_part(desc):
    """Return only the text describing the play itself."""
    # if replay reversed the call, the description holds the original ruling first and the
    # corrected one after "REVERSED." Only the corrected version counts.
    if "REVERSED." in desc:
        desc = desc.split("REVERSED.", 1)[1]
    # everything before the first mention of a penalty is the play itself
    return re.split(r"PENALTY|Penalty on", desc, maxsplit = 1)[0]

null["action"] = null["desc"].apply(on_field_part)

def classify(action):
    """Sort the on-field part of a nullified play into a type."""
    if re.search(r"INTERCEPTED", action):
        return "interception"
    if re.search(r"FUMBLES|Aborted", action):
        return "fumble" # manual review
    if re.search(r"sacked", action):
        return "sack"
    if re.search(r"pass incomplete", action):
        return "incomplete"
    if re.search(r" pass ", action):
        return "completion"
    if re.search(r"scrambles", action):
        return "scramble"
    if re.search(r" for (-?\d+ yards?|no gain)", action):
        return "run"
    return "pre-snap" # no play happened

def parse_yards(action):
    """Pull the yards gained from the on-field part. Returns NaN if there are none."""
    action = re.split(r"INTERCEPTED|FUMBLES", action)[0]
    m = re.search(r" for (-?\d+) yards?| for no gain", action)
    if m is None:
        return np.nan
    return 0 if m.group(1) is None else int(m.group(1))

null["action_type"] = null["action"].apply(classify)
null["parsed_yards"] = null["action"].apply(parse_yards)
null["td_nullified"] = null["desc"].str.contains("TOUCHDOWN NULLIFIED", na = False)

null["action_type"].value_counts()

action_type
pre-snap        721
incomplete      392
run             229
completion      212
sack             52
scramble         38
interception     30
fumble            5
Name: count, dtype: int64

In [3]:
# penalty assignment
null["penalty_on_offense"] = null["penalty_team"] == null["posteam"]
pd.crosstab(null["action_type"], null["penalty_on_offense"], margins = True)

penalty_on_offense,False,True,All
action_type,,,
completion,42,170,212
fumble,4,1,5
incomplete,324,68,392
interception,30,0,30
pre-snap,124,597,721
run,46,183,229
sack,48,4,52
scramble,1,37,38
All,619,1060,1679


## Null Ledger
Each parsed play turns into ledger rows in the same format as the ledger in 02, using the stats the player would have earned.

In [4]:
KEYS = ["season", "week", "game_id", "play_id"]
LABELS = ["action_type", "td_nullified", "penalty_on_offense", "penalty_type", "penalty_player_id"]

def ledger_rows(df, player_col, stat_name, value):
    """Same helper as 02, with the penalty labels carried along."""
    out = df[KEYS + LABELS + [player_col]].rename(columns={player_col: "player_id"})
    out["stat"] = stat_name
    out["value"] = df[value].values if isinstance(value, str) else value
    return out.dropna(subset=["player_id"])

comp = null[null["action_type"] == "completion"]
run = null[null["action_type"] == "run"]
scram = null[null["action_type"] == "scramble"]
ints = null[null["action_type"] == "interception"]
inc = null[null["action_type"] == "incomplete"]
two_pt = null[(null["action_type"] == "two_point") & null["desc"].str.contains("ATTEMPT SUCCEEDS")]

nullified = pd.concat([
    ledger_rows(comp, "passer_id", "passing_yards", "parsed_yards"),
    ledger_rows(comp[comp["td_nullified"]], "passer_id", "passing_tds", 1),
    ledger_rows(comp, "receiver_id", "receptions", 1),
    ledger_rows(comp, "receiver_id", "receiving_yards", "parsed_yards"),
    ledger_rows(comp[comp["td_nullified"]], "receiver_id", "receiving_tds", 1),
    ledger_rows(run, "rusher_id", "rushing_yards", "parsed_yards"),
    ledger_rows(run[run["td_nullified"]], "rusher_id", "rushing_tds", 1),
    ledger_rows(scram, "passer_id", "rushing_yards", "parsed_yards"),
    ledger_rows(scram[scram["td_nullified"]], "passer_id", "rushing_tds", 1),
    ledger_rows(ints, "passer_id", "interceptions", 1),
    # successful two-point tries (same crediting as notebook 2)
    ledger_rows(two_pt, "passer_id", "two_pt_conversions", 1),
    ledger_rows(two_pt, "receiver_id", "two_pt_conversions", 1),
    ledger_rows(two_pt, "rusher_id", "two_pt_conversions", 1),
    # targets (follows nflverse: completions, incompletions, and interceptions all count
    # as a target for the intended receiver; throwaways have no receiver and drop out)
    ledger_rows(comp, "receiver_id", "targets", 1),
    ledger_rows(inc, "receiver_id", "targets", 1),
    ledger_rows(ints, "receiver_id", "targets", 1),
], ignore_index=True)


nullified["points"] = nullified["value"] * nullified["stat"].map(SCORING)
# Targets aren't worth points on their own, so they get 0 instead of being left blank
nullified.loc[nullified["stat"] == "targets", "points"] = 0
# A stat missing from SCORING is likely a typo in a stat name
unscored = set(nullified["stat"]) - set(SCORING) - {"targets"}
if unscored:
    print(f"WARNING: no scoring value for {sorted(unscored)}")
# A blank value means the parser couldn't read the yards on that play
if nullified["value"].isna().any():
    print(f"WARNING: {nullified['value'].isna().sum()} ledger rows have no value")
nullified["penalty_on_self"] = nullified["player_id"] == nullified["penalty_player_id"]
print(f"Nullified ledger rows: {len(nullified):,}")
nullified.head()

Nullified ledger rows: 1,549


,season,week,game_id,play_id,action_type,td_nullified,penalty_on_offense,penalty_type,penalty_player_id,player_id,stat,value,points,penalty_on_self
0,2026,1,2026_01_ARI_LAC,637.0,completion,False,True,Offensive Holding,00-0035258,00-0033119,passing_yards,6.0,0.24,False
1,2026,1,2026_01_ARI_LAC,1363.0,completion,False,True,Illegal Block Above the Waist,00-0037009,00-0036355,passing_yards,5.0,0.20,False
2,2026,1,2026_01_ARI_LAC,3784.0,completion,False,True,Offensive Holding,00-0041392,00-0036355,passing_yards,4.0,0.16,False
3,2026,1,2026_01_ATL_PIT,1872.0,completion,False,True,Offensive Holding,00-0035630,00-0033662,passing_yards,8.0,0.32,False
4,2026,1,2026_01_ATL_PIT,2046.0,completion,False,True,Offensive Pass Interference,00-0035640,00-0023459,passing_yards,19.0,0.76,False


## Test 1: Run the parser on plays that counted
The descriptions of normal plays use the same wording as nullified ones, but for normal plays nflverse already knows the right answer. Running the same parser on them shows how often it reads the yards correctly. This doesn't replace the hand check, but it tests thousands of plays at once.

In [5]:
real = pbp[pbp["play_type"].isin(["pass", "run"]) & (pbp["penalty"] != 1)
           & (pbp["two_point_attempt"] != 1) & (pbp["lateral_reception"] != 1)
           & (pbp["lateral_rush"] != 1)].copy()
real["action"] = real["desc"].apply(on_field_part)
real["action_type"] = real["action"].apply(classify)
real["parsed_yards"] = real["action"].apply(parse_yards)

real_comp = real[real["action_type"] == "completion"]
real_rush = real[real["action_type"].isin(["run", "scramble"])]

comp_acc = (real_comp["parsed_yards"] == real_comp["receiving_yards"]).mean()
rush_acc = (real_rush["parsed_yards"] == real_rush["rushing_yards"]).mean()

# plays with fumbles handled separately
typed = real[real["action_type"] != "fumble"]
type_acc = ((typed["action_type"] == "completion") == (typed["complete_pass"] == 1)).mean()

print(f"Completions: yards match on {comp_acc:.2%} of {len(real_comp):,} plays")
print(f"Runs and scrambles: yards match on {rush_acc:.2%} of {len(real_rush):,} plays")
print(f"Completion vs. not: classified correctly on {type_acc:.2%} of {len(typed):,} plays")

Completions: yards match on 100.00% of 6,648 plays
Runs and scrambles: yards match on 100.00% of 8,198 plays
Completion vs. not: classified correctly on 100.00% of 19,331 plays


## Test 2: Hand Check
A random sample of parsed nullified plays saved as CSV and checked by hand in Excel.

In [6]:
action_plays = null[null["action_type"].isin(["completion", "run", "scramble", "interception"])]
sample = action_plays.sample(min(50, len(action_plays)), random_state=1906)

review = sample[["game_id", "play_id", "desc", "action_type", "parsed_yards", "td_nullified",
                 "passer", "receiver", "rusher"]].copy()
review["ok_type"] = ""
review["ok_yards"] = ""
review["ok_td"] = ""

# Only written once, so rerunning the notebook never erases your checks
if not os.path.exists("data/hand_check_nullified.csv"):
    review.to_csv("data/hand_check_nullified.csv", index=False)
    print(f"Saved {len(review)} plays to data/hand_check_nullified.csv")
else:
    print(f"data/hand_check_nullified.csv already exists, so it was left alone")

data/hand_check_nullified.csv already exists, so it was left alone


In [7]:
# ran afterward
checked = pd.read_csv("data/hand_check_nullified.csv")
for col in ["ok_type", "ok_yards", "ok_td"]:
    filled = pd.to_numeric(checked[col], errors="coerce").dropna()
    if len(filled):
        print(f"{col}: {filled.mean():.0%} correct ({len(filled)} checked)")
    else:
        print(f"{col}: not filled in yet")

ok_type: 98% correct (50 checked)
ok_yards: 98% correct (50 checked)
ok_td: 100% correct (50 checked)


## Manual Review: Fumbles
Plays with a fumble are not parsed automatically. Fumbles themselves are relatively rare, nullified defense-recovered fumbles even less common. If any occur, the number will be few enough to be reviewed individually.

In [8]:
null[null["action_type"] == "fumble"][["game_id", "play_id", "desc"]]

,game_id,play_id,desc
3872,2026_02_JAX_DEN,261.0,"(11:13) 64-W.Milum reported in as eligible. 16-T.Lawrence sacked at JAX 34 for -9 yards (15-N.Bonitto). FUMBLES (15-N.Bonitto), RECOVERED by DEN-49-A.Singleton at JAX 30. 49-A.Singleton to JAX 21 ..."
11126,2026_04_TEN_BAL,3705.0,"(9:39) (Shotgun) 5-T.Huntley Aborted. 71-O.Ioane FUMBLES at BAL 20, recovered by BAL-5-T.Huntley at BAL 22. 5-T.Huntley to BAL 22 for no gain (93-N.Stackhouse, 7-O.Oladejo). ** Injury Update: TEN-..."
16747,2026_07_DAL_PHI,850.0,"(3:18) 16-T.McKee FUMBLES (Aborted) at PHI 44, RECOVERED by DAL-92-Q.Williams at PHI 43. PENALTY on DAL-92-Q.Williams, Defensive Offside, 5 yards, enforced at PHI 44 - No Play."
26009,2026_10_SF_DAL,3463.0,"(11:03) (No Huddle, Shotgun) 4-D.Prescott sacked at DAL 33 for -5 yards (98-M.Williams). FUMBLES (98-M.Williams) [98-M.Williams], recovered by DAL-88-C.Lamb at DAL 30. PENALTY on SF-2-D.Lenoir, De..."
28396,2026_11_TEN_DAL,2748.0,"(8:16) (No Huddle) 1-C.Ward scrambles left end to DAL 46 for 8 yards (45-J.Barron). FUMBLES (45-J.Barron), ball out of bounds at DAL 45. PENALTY on TEN-76-C.Zavala, Offensive Holding, 10 yards, en..."


In [9]:
MANUAL_FUMBLES = [
    ("2026_02_JAX_DEN", 261, "00-0036971", "T.Lawrence sack fumble, recovered by DEN"),
    ("2026_07_DAL_PHI", 850, "00-0038400", "T.McKee aborted snap, recovered by DAL"),
]

manual = pd.DataFrame(MANUAL_FUMBLES, columns = ["game_id", "play_id", "player_id", "note"])
manual = manual.merge(null[KEYS + LABELS], on = ["game_id", "play_id"], how = "left")
manual["stat"] = "fumbles_lost"
manual["value"] = 1
manual["points"] = manual["value"] * SCORING["fumbles_lost"]
manual["penalty_on_self"] = manual["player_id"] == manual["penalty_player_id"]

nullified = nullified[nullified["stat"] != "fumbles_lost"]
nullified = pd.concat([nullified, manual.drop(columns = "note")], ignore_index = True)
print(f"Added {len(manual)} manual fumbles. Nullified ledger rows: {len(nullified):,}")
manual[["game_id", "play_id", "note", "penalty_type", "penalty_on_offense", "points"]]

Added 2 manual fumbles. Nullified ledger rows: 1,551


,game_id,play_id,note,penalty_type,penalty_on_offense,points
0,2026_02_JAX_DEN,261,"T.Lawrence sack fumble, recovered by DEN",Illegal Contact,False,-2
1,2026_07_DAL_PHI,850,"T.McKee aborted snap, recovered by DAL",Defensive Offside,False,-2


## First look
Total points wiped out per player, split by who committed the penalty. Names from official stats table.

In [10]:
names = pd.read_parquet(f"data/player_stats_{CURRENT_SZN}.parquet")
names = names[["player_id", "player_display_name", "position"]].drop_duplicates("player_id")

lost = (
    nullified.groupby(["player_id", "penalty_on_offense", "penalty_on_self"], as_index = False)["points"].sum()
    .merge(names, on = "player_id", how = "left")
)
by_player = lost.pivot_table(index = ["player_display_name", "position"],
                             columns = "penalty_on_offense", values = "points",
                             aggfunc = "sum", fill_value = 0)
by_player.columns = ["by_defense" if c is False else "by_offense" for c in by_player.columns]
by_player["total"] = by_player.sum(axis=1)
by_player.sort_values("total", ascending = False).head(20)

,,by_defense,by_offense,total
player_display_name,position,,,
Chris Godwin Jr.,WR,0.00,25.30,25.30
Bucky Irving,RB,1.40,22.20,23.60
Bhayshul Tuten,RB,0.00,17.80,17.80
Blake Corum,RB,0.00,17.60,17.60
Antonio Williams,WR,0.00,15.70,15.70
Jaxon Smith-Njigba,WR,0.00,15.40,15.40
Baker Mayfield,QB,-1.76,14.80,13.04
Chase Brown,RB,0.00,12.80,12.80
Chuba Hubbard,RB,3.60,9.10,12.70


In [11]:
season_tag = str(CURRENT_SZN)
nullified.to_parquet(f"data/nullified_ledger_{season_tag}.parquet", index = False)
null.drop(columns = ["action"]).to_parquet(f"data/nullified_plays_{season_tag}.parquet", index = False)
print("saved")

saved
